# Variant Calling
Variant calling on RNA-seq data to determine familiar structure of my samples, following [GATK Best Practices Workflow for RNAseq short variant discover (SNPs + indels)](https://gatk.broadinstitute.org/hc/en-us/articles/360035531192-RNAseq-short-variant-discovery-SNPs-Indels)

Most of the processing steps will be run as batch jobs from the `/work` directory. Scripts are below.


## 1. Mapping to the reference with [STAR](https://github.com/alexdobin/STAR) aligner
see [STAR manual](https://github.com/alexdobin/STAR/blob/master/doc/STARmanual.pdf)

### A. Generate genome index files

In [ ]:
# get GTF annotation file
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/053/477/285/GCF_053477285.1_ASM5347728v1/GCF_053477285.1_ASM5347728v1_genomic.gtf.gz

gunzip GCF_053477285.1_ASM5347728v1_genomic.gtf.gz 

In [ ]:
#!/bin/bash
#SBATCH --job-name=star_index
#SBATCH -c 16
#SBATCH --nodes=1
#SBATCH --mem=80G
#SBATCH -p cpu
#SBATCH -t 08:00:00
#SBATCH -o star_index_%j.out
#SBATCH -e star_index_%j.err
#SBATCH --mail-type=END,FAIL

#############################
# MODULES
module load star/2.7.11a

#############################
# USER PARAMETERS
THREADS=16

GENOME_FA=/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/GCF_053477285.1_ASM5347728v1_genomic.fna
GTF=/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/GCF_053477285.1_ASM5347728v1_genomic.gtf

OUTDIR=/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ce24_rnaseq/gatk/star_align

# For RNA-seq:
# set to (read length - 1)
SJDB_OVERHANG=149   # for 150bp reads

#############################
# STAR INDEX BUILD

STAR \
  --runThreadN ${THREADS} \
  --runMode genomeGenerate \
  --genomeDir ${OUTDIR} \
  --genomeFastaFiles ${GENOME_FA} \
  --sjdbGTFfile ${GTF} \
  --sjdbOverhang ${SJDB_OVERHANG} \
  --genomeSAindexNbases 14

### B. Mapping reads to the genome
need to do a 2-pass approach to account for both annotated and unannotated splice junctions (specific boundaries where introns are removed and exons are joined together)

I was getting errors running STAR using `module load star` - the version of STAR loaded in unity has conflicts, so I created an enviornment instead:

In [ ]:
module load conda/latest

conda create -n star_env -c conda-forge -c bioconda star=2.7.11a

conda activate star_env

For mapping with `STAR`, it's recommended to do two-pass mapping - this allows for detection of more splices reads mapping to novel junctions

In [ ]:
#!/bin/bash
#SBATCH --job-name=star_align
#SBATCH -c 16
#SBATCH --mem=60G
#SBATCH -p cpu
#SBATCH -t 12:00:00
#SBATCH --array=1-120
#SBATCH -o star_%A_%a.out
#SBATCH -e star_%A_%a.err
#SBATCH --nodes=1

module load conda/latest
conda activate star_env

#############################
# INPUTS
#############################

SAMPLE_SHEET=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all/samples.txt
INDEX=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/star_align/  
OUTDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/star_bams/

#############################
# GET THIS ARRAY SAMPLE
#############################

LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" ${SAMPLE_SHEET})

SAMPLE=$(echo $LINE | awk '{print $1}')
R1=$(echo $LINE | awk '{print $2}')
R2=$(echo $LINE | awk '{print $3}')

#############################
# RUN STAR
#############################

STAR \
  --runThreadN 16 \
  --genomeDir ${INDEX} \
  --readFilesIn ${R1} ${R2} \
  --readFilesCommand zcat \
  --outFileNamePrefix ${OUTDIR}/${SAMPLE}. \
  --outSAMtype BAM SortedByCoordinate \
  --quantMode GeneCounts \
  --twopassMode Basic

## 2. Data cleanup

using [`MarkedDuplicates`](https://gatk.broadinstitute.org/hc/en-us/articles/13832748517275-MarkDuplicates-Picard) from Picard to deduplicate reads 

In [ ]:
#!/bin/bash 
#SBATCH -c 4
#SBATCH --mem=16G
#SBATCH -t 4:00:00
#SBATCH --array=1-120
#SBATCH -o markdup_%A_%a.out
#SBATCH -e markdup_%A_%a.err

module load gatk/4.5.0.0

#############################
# INPUTS
#############################

SAMPLE_SHEET=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all/samples.txt
INDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/star_bams
OUTDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/markDupes

#############################
# GET SAMPLE FROM ARRAY
#############################

LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" ${SAMPLE_SHEET})
SAMPLE=$(echo $LINE | awk '{print $1}')

#############################
# RUN MARKDUPLICATES
#############################

gatk MarkDuplicates \
    -I ${INDIR}/${SAMPLE}.Aligned.sortedByCoord.out.bam \
    -O ${OUTDIR}/${SAMPLE}.dedup.bam \
    -M ${OUTDIR}/${SAMPLE}.dedup.metrics.txt \
    --CREATE_INDEX true \
    --VALIDATION_STRINGENCY SILENT

## 3. [SplitNCigarReads](https://gatk.broadinstitute.org/hc/en-us/articles/13832774383643-SplitNCigarReads)
Splits reads that contain Ns in their cigar string (e.g. spanning splicing events in RNAseq data) - basically, creates exon-specific sequences

In [ ]:
#!/bin/bash
#SBATCH --job-name=split_cigar
#SBATCH -c 4
#SBATCH --mem=100G
#SBATCH -p cpu
#SBATCH -t 4:00:00
#SBATCH --array=1-120
#SBATCH -o split_%A_%a.out
#SBATCH -e split_%A_%a.err
#SBATCH --nodes=1

module load gatk/4.5.0.0

#############################
# INPUTS
#############################

SAMPLE_SHEET=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all/samples.txt
REF=/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/GCF_053477285.1_ASM5347728v1_genomic.fna
INDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/markDupes
OUTDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/gatk/splitNCigarReads

mkdir -p ${OUTDIR}

#############################
# GET SAMPLE NAME
#############################

LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" ${SAMPLE_SHEET})

SAMPLE=$(echo ${LINE} | awk '{print $1}')

echo "Processing ${SAMPLE}"

#############################
# CHECK INPUT
#############################

INPUT=${INDIR}/${SAMPLE}.dedup.bam
OUTPUT=${OUTDIR}/${SAMPLE}.split.bam

if [ ! -f ${INPUT} ]; then
    echo "ERROR: Missing BAM ${INPUT}"
    exit 1
fi

#############################
# SPLIT NCIGAR READS
#############################

gatk SplitNCigarReads \
    -R ${REF} \
    -I ${INPUT} \
    -O ${OUTPUT}